# 01 — Data exploration

**Goal:** Get familiar with the datasets before working with models.

Datasets:
- **mmlu_ukr** — Ukrainian multiple-choice questions across academic subjects.
- **ifeval_ukr** — Ukrainian prompts used to evaluate how well a model follows specific instructions.
- **XL-Sum** — News articles paired with summaries; this notebook uses the Ukrainian subset.
- **wiki-instruction-dialogs** — Instruction–response examples and conversations based on Wikipedia content.

In [283]:
import pandas as pd
import re
import json
import numpy as np
from datasets import load_dataset

# mmlu_ukr

In [251]:
splits = {'test': 'data/test-00000-of-00001.parquet', 'dev': 'data/dev-00000-of-00001.parquet', 'validation': 'data/validation-00000-of-00001.parquet'}
test_df = pd.read_parquet("hf://datasets/INSAIT-Institute/mmlu_ukr/" + splits["test"])
dev_df = pd.read_parquet("hf://datasets/INSAIT-Institute/mmlu_ukr/" + splits["dev"])
val_df = pd.read_parquet("hf://datasets/INSAIT-Institute/mmlu_ukr/" + splits["validation"])

In [284]:
print(test_df.shape)
print(dev_df.shape)
print(val_df.shape)

(5399, 5)
(285, 4)
(5399, 5)


In [285]:
print(dev_df.head())
print("\nRaw dtypes:\n", dev_df.dtypes)
print("\nNumber of missing values in each column:\n", dev_df.isna().sum())
print("\nNumber of unique values in each column (excluding choices):\n", dev_df.drop(columns="choices").nunique())
print("\nNumber of subjects and examples:\n",  dev_df["subject"].value_counts())
print("\nNumber of duplicated questions:", dev_df["question"].duplicated().sum())

                                            question           subject  \
0  Знайдіть всі c у Z_3 такі, що Z_3[x]/(x^2 + c)...  abstract_algebra   
1  Твердження 1 | Якщо aH є елементом фактор-груп...  abstract_algebra   
2  Твердження 1 | Кожен елемент групи генерує цик...  abstract_algebra   
3  Твердження 1 | Кожна функція з кінцевої множин...  abstract_algebra   
4                 Знайдіть характеристику кільця 2Z.  abstract_algebra   

                                             choices  answer  
0                                       [0, 1, 2, 3]       1  
1  [Правда, Правда, Неправда, Неправда, Правда, Н...       1  
2  [Правда, Правда, Неправда, Неправда, Правда, Н...       2  
3  [Правда, Правда, Неправда, Неправда, Правда, Н...       0  
4                                     [0, 3, 12, 30]       0  

Raw dtypes:
 question       str
subject        str
choices     object
answer       uint8
dtype: object

Number of missing values in each column:
 question    0
subject     0
ch

In [286]:
print(dev_df["answer"].describe())

count    285.000000
mean       1.561404
std        1.126154
min        0.000000
25%        1.000000
50%        2.000000
75%        3.000000
max        3.000000
Name: answer, dtype: float64


In [287]:
dev_df["answer"] = dev_df["answer"].astype("uint8")

In [288]:
print(dev_df["choices"][:5])

0                                         [0, 1, 2, 3]
1    [Правда, Правда, Неправда, Неправда, Правда, Н...
2    [Правда, Правда, Неправда, Неправда, Правда, Н...
3    [Правда, Правда, Неправда, Неправда, Правда, Н...
4                                       [0, 3, 12, 30]
Name: choices, dtype: object


In [289]:
row = dev_df.iloc[0]

print("Question: \n", row["question"])
print("Choices: \n", row["choices"])
print("Answer index:", row["answer"])

Question: 
 Знайдіть всі c у Z_3 такі, що Z_3[x]/(x^2 + c) є полем.
Choices: 
 ['0' '1' '2' '3']
Answer index: 1


In [258]:
def mmlu_prompt_formation(record):
    answer_options = []

    for i, choice in enumerate(record["choices"]):
        answer_options.append(f"{i}. {choice}")

    answer_options_combined = "\n".join(answer_options)
    question = record['question']
    question = re.sub(r" {2,}", " ", question)

    prompt = (
        f"Тема: {record['subject']}\n"
        f"Запитання: {question}\n"
        f"Варіанти відповіді:\n{answer_options_combined}\n"
        "Відповідай лише номером правильного варіанта:"
    )

    return prompt, int(record["answer"])

In [259]:
prompt, answer = mmlu_prompt_formation(row)
print("Prompt: \n", prompt)
print("Correct option index:", answer)

Prompt: 
 Тема: abstract_algebra
Запитання: Знайдіть всі c у Z_3 такі, що Z_3[x]/(x^2 + c) є полем.
Варіанти відповіді:
0. 0
1. 1
2. 2
3. 3
Відповідай лише номером правильного варіанта:
Correct option index: 1


# ifeval_ukr

In [292]:
df = pd.read_parquet("hf://datasets/INSAIT-Institute/ifeval_ukr/data/train-00000-of-00001.parquet")

In [293]:
print(df.head())
print("\nRaw data types:\n", df.dtypes)

    key                                             prompt  \
0  1000  Напишіть підсумок статті з Вікіпедії "https://...   
1  1001  Я планую подорож до Японії і хотів би, щоб ти ...   
2  1005  Напишіть резюме для випускника середньої школи...   
3  1012  Напишіть електронний лист моєму начальнику, по...   
4  1019  Дано речення: "Двоє молодих хлопців з іграшков...   

                                 instruction_id_list  \
0  [punctuation:no_comma, detectable_format:numbe...   
1                             [punctuation:no_comma]   
2           [detectable_content:number_placeholders]   
3  [combination:repeat_prompt, detectable_format:...   
4                    [change_case:english_lowercase]   

                                              kwargs  
0  [{'num_highlights': None, 'relation': None, 'n...  
1  [{'num_highlights': None, 'relation': None, 'n...  
2  [{'num_highlights': None, 'relation': None, 'n...  
3  [{'num_highlights': None, 'relation': None, 'n...  
4  [{'num_highl

In [294]:
print(df["key"][:5])

0    1000
1    1001
2    1005
3    1012
4    1019
Name: key, dtype: int64


In [295]:
row = df.iloc[0]

print("Key: \n", row['key'])
print("Prompt: \n", row["prompt"])
print("Instructions: \n", row["instruction_id_list"])
print("Additional instructions: \n", row["kwargs"])

Key: 
 1000
Prompt: 
 Напишіть підсумок статті з Вікіпедії "https://en.wikipedia.org/wiki/Raymond_III,_Count_of_Tripoli" обсягом понад 300 слів. Не використовуйте коми та виділіть принаймні 3 розділи з заголовками у форматі markdown, наприклад *виділений розділ частина 1*, *виділений розділ частина 2*, *виділений розділ частина 3*.
Instructions: 
 ['punctuation:no_comma' 'detectable_format:number_highlighted_sections'
 'length_constraints:number_words']
Additional instructions: 
 [{'num_highlights': None, 'relation': None, 'num_words': None, 'num_placeholders': None, 'prompt_to_repeat': None, 'num_bullets': None, 'section_spliter': None, 'num_sections': None, 'capital_relation': None, 'capital_frequency': None, 'keywords': None, 'num_paragraphs': None, 'language': None, 'let_relation': None, 'letter': None, 'let_frequency': None, 'end_phrase': None, 'forbidden_words': None, 'keyword': None, 'frequency': None, 'num_sentences': None, 'postscript_marker': None, 'first_word': None, 'nth_pa

In [296]:
print("\nNumber of missing values in each column:\n", df.isna().sum())
print("\nNumber of unique prompts in each column:\n", df.prompt.nunique())


Number of missing values in each column:
 key                    0
prompt                 0
instruction_id_list    0
kwargs                 0
dtype: int64

Number of unique prompts in each column:
 541


In [299]:
def decode(value):
    if isinstance(value, np.ndarray):
        return decode(value.tolist())
    elif isinstance(value, np.generic):
        return value.item()
    elif isinstance(value, dict):
        return {key: decode(item) for key, item in value.items()}
    elif isinstance(value, (list, tuple)):
        return [decode(item) for item in value]
    return value

In [302]:
columns = ["prompt", "instruction_id_list", "kwargs"]
check_df = df[columns].copy()

for column in ["instruction_id_list", "kwargs"]:
    check_df[column] = check_df[column].map(lambda value: json.dumps(decode(value), ensure_ascii=False, sort_keys=True))

print("Number of repeated records:", check_df.duplicated().sum())

Number of repeated records: 0


In [298]:
def ifeval_prompt_formation(record):
    prompt = record["prompt"]
    prompt = re.sub(r" {2,}", " ", prompt)
    instruction_kwargs_pairs = []

    for i, params in zip(record["instruction_id_list"], record["kwargs"]):
        relevant_kwargs = {}
        for k, v in params.items():
            if v is not None:
                relevant_kwargs[k] = v

        instruction_kwargs_pairs.append({"instruction": i, "kwargs": relevant_kwargs,})

    return prompt, instruction_kwargs_pairs

In [266]:
prompt, checks = ifeval_prompt_formation(row)

print("Prompt: \n", prompt)
print("Instructions to check the answer:")
for check in checks:
    print(check)

Prompt: 
 Напишіть підсумок статті з Вікіпедії "https://en.wikipedia.org/wiki/Raymond_III,_Count_of_Tripoli" обсягом понад 300 слів. Не використовуйте коми та виділіть принаймні 3 розділи з заголовками у форматі markdown, наприклад *виділений розділ частина 1*, *виділений розділ частина 2*, *виділений розділ частина 3*.
Instructions to check the answer:
{'instruction': 'punctuation:no_comma', 'kwargs': {}}
{'instruction': 'detectable_format:number_highlighted_sections', 'kwargs': {'num_highlights': 3.0}}
{'instruction': 'length_constraints:number_words', 'kwargs': {'relation': 'at least', 'num_words': 300.0}}


# Xlsum

In [305]:
ds = load_dataset("json", data_files={
        "train": "../data/raw/ukrainian_XLSum_v2.0/ukrainian_train.jsonl",
        "validation": "../data/raw/ukrainian_XLSum_v2.0/ukrainian_val.jsonl",
        "test": "../data/raw/ukrainian_XLSum_v2.0/ukrainian_test.jsonl",
    },
)

train_df, val_df, test_df = ds["train"].to_pandas(), ds["validation"].to_pandas(), ds["test"].to_pandas()

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(43201, 5)
(5399, 5)
(5399, 5)


In [306]:
print(train_df.head())

                         id  \
0         features-41015786   
1     031017_germanyreform1   
2  060724_chavez_belarus_sp   
3     040404_spain_bombings   
4            media-45563874   

                                                 url  \
0    https://www.bbc.com/ukrainian/features-41015786   
1  https://www.bbc.com/ukrainian/news/story/2003/...   
2  https://www.bbc.com/ukrainian/news/story/2006/...   
3  https://www.bbc.com/ukrainian/news/story/2004/...   
4       https://www.bbc.com/ukrainian/media-45563874   

                                               title  \
0                Стінопис: від Філадельфії до Рабата   
1    Німецький парламент ухвалив непопулярні реформи   
2       Чавес закликав Лукашенка втілити ідеї Леніна   
3           Загинув організатор мадридських вибухів?   
4  Джемілєв: "Помирати до звільнення Криму - це д...   

                                             summary  \
0  Вже кілька років на київських багатоповерхівка...   
1  Нижня палата німецького 

In [307]:
print("\nRaw dtypes:\n", train_df.dtypes)
print("\nNumber of missing values in each column:\n", train_df.isna().sum())
print("\nNumber of unique values in each column:\n", train_df.nunique())


Raw dtypes:
 id         str
url        str
title      str
summary    str
text       str
dtype: object

Number of missing values in each column:
 id         0
url        0
title      0
summary    0
text       0
dtype: int64

Number of unique values in each column:
 id         43201
url        43201
title      43128
summary    42843
text       43121
dtype: int64


In [308]:
row = train_df.iloc[0]

print(row["id"])
print(row["url"])
print(row["title"])
print(row["summary"])
print(row["text"])

features-41015786
https://www.bbc.com/ukrainian/features-41015786
Стінопис: від Філадельфії до Рабата
Вже кілька років на київських багатоповерхівках з'являються сюжети з неприборканої уяви майстрів графіті з усього світу. Лише цього серпня принаймні на три прикрашені живописом стіни в столиці стало більше: "Просте щастя" - обличчя усміхненого чорношкірого хлопченяти, меморіальний портрет виконавця Linkin Park Честера Беннінгтона і філософська фреска "Розум, тіло і душа".
Якщо для Києва "наскельне мистецтво" явище відносно нове, то на Заході - майже класичне. От лише деякі з графіті, які потрапили в фотооб'єктив мандрівника Андрія Кондратьєва. За великим рахунком, зображення у печері Ласко у Франції, яким, як вважають вчені, приблизно 17 тисяч років, - теж графіті. Але сучасна версія стінопису зародилася в Сполучених Штатах - у Нью-Йорку і Філадельфії. І в останній еволюціонувала і стала частиною міського ДНК. Там скупчено більше муралів, ніж будь-де в країні. Адже ще у 1984 році затве

In [309]:
print("Number of duplicated records:", train_df.duplicated().sum())

Number of duplicated records: 0


In [271]:
def xlsum_prompt_formation(record):
    title, url, text = record["title"], record["url"], record["text"]
    title = re.sub(r" {2,}", " ", title)
    text = re.sub(r" {2,}", " ", text)

    prompt = (
        "Підсумуй цю статтю:\n"
        f"Назва: {title}\n"
        f"Посилання: {url}\n"
        f"Текст статті: \n{text}\n"
    )

    return prompt, record["summary"]

In [272]:
prompt, summary = xlsum_prompt_formation(row)
print("Prompt: \n", prompt)
print("Summarized version: \n", summary)

Prompt: 
 Підсумуй цю статтю:
Назва: Стінопис: від Філадельфії до Рабата
Посилання: https://www.bbc.com/ukrainian/features-41015786
Текст статті: 
Якщо для Києва "наскельне мистецтво" явище відносно нове, то на Заході - майже класичне. От лише деякі з графіті, які потрапили в фотооб'єктив мандрівника Андрія Кондратьєва. За великим рахунком, зображення у печері Ласко у Франції, яким, як вважають вчені, приблизно 17 тисяч років, - теж графіті. Але сучасна версія стінопису зародилася в Сполучених Штатах - у Нью-Йорку і Філадельфії. І в останній еволюціонувала і стала частиною міського ДНК. Там скупчено більше муралів, ніж будь-де в країні. Адже ще у 1984 році затвердили програму, однією з цілей якої було забезпечити спраглих до експериментальної на той час творчості митців цегляним "полотном", на якому вони могли творити. Результат - близько трьох тисяч мистецьких робіт. Окрему нішу серед вуличних художників Філадельфії займає Ісаї Загар, який працює у жанрі мозаїки. Він трансформував баг

# wiki-instruction-dialogs

In [310]:
ds = load_dataset("lapa-llm/wiki-instruction-dialogs")
train_df = ds["train"].to_pandas()

print(train_df.shape)

(30684, 5)


In [311]:
print(train_df.head())

             task                                        instruction  \
0             ner  Витягни всі згадки про людей, місця, компанії ...   
1             ner  Зроби анотацію іменованих сутностей у поданому...   
2  simplification                  Перекажи суть коротко і зрозуміло   
3  simplification                Напиши резюме тексту своїми словами   
4  simplification                  Перекажи суть коротко і зрозуміло   

                                               input  \
0  Геогра́фія, або заст. земле́пис (від  — опис З...   
1  Уперше термін «географія» був запропонований д...   
2  Основне завдання сучасної географії полягає у ...   
3  Природничо-географічні (фізико-географічні) на...   
4  До суспільно-географічних (соціально-економічн...   

                                              output  \
0  Люди: людина\nМісця: Земля, регіони, країни\nК...   
1  *   **Ератосфен** - особа (географ)\n*   **дав...   
2  Сучасна географія вивчає закони розміщення та ...   
3  При

In [312]:
print("\nRaw dtypes:\n", train_df.dtypes)
print("\nNumber of missing values in each column:\n", train_df.isna().sum())
print("\nNumber of unique values in each column (except conversations):\n", train_df.drop(columns="conversations").nunique())


Raw dtypes:
 task                str
instruction         str
input               str
output              str
conversations    object
dtype: object

Number of missing values in each column:
 task             0
instruction      0
input            0
output           0
conversations    0
dtype: int64

Number of unique values in each column (except conversations):
 task               4
instruction       78
input          29663
output         30272
dtype: int64


In [313]:
print(train_df["input"])

0        Геогра́фія, або заст. земле́пис (від  — опис З...
1        Уперше термін «географія» був запропонований д...
2        Основне завдання сучасної географії полягає у ...
3        Природничо-географічні (фізико-географічні) на...
4        До суспільно-географічних (соціально-економічн...
                               ...                        
30679    Для двох множин також можна ввести операцію «в...
30680    За домовленістю усі обговорювані множини вважа...
30681    * Якщо  — множина цілих чисел, то доповнення ї...
30682    Симетричною різницею множин A та B", що познач...
30683    Нову множину можна побудувати, пов'язуючи коже...
Name: input, Length: 30684, dtype: str


In [314]:
row = train_df.iloc[0]

print("Task: ", row["task"], "\n")
print("Instruction: ", row["instruction"], "\n")
print("Input: ", row["input"], "\n")
print("Output: ", row["output"], "\n")

Task:  ner 

Instruction:  Витягни всі згадки про людей, місця, компанії та дати з тексту 

Input:  Геогра́фія, або заст. земле́пис (від  — опис Землі; де  — Земля і  — писати, описувати) — система наук, що вивчає географічну оболонку Землі (епігеосферу), її просторову природну і соціально-економічну різноманітність, господарство і населення планети, окремих її регіонів та країн, а також зв'язки між природним середовищем і діяльністю людини. В сучасному розумінні поняття «географія» заміщено поняттям «географічні науки». 

Output:  Люди: людина
Місця: Земля, регіони, країни
Компанії: —
Дати: — 



In [315]:
print("Repeated input:", train_df.duplicated(subset=["input"]).sum())
print("Repeated input + instruction:", train_df.duplicated(subset=["instruction", "input"]).sum())
print("Same examples:", train_df.duplicated(subset=["instruction", "input", "output"]).sum())

train_df = train_df.drop_duplicates(subset=["instruction", "input", "output"]).copy()
print(train_df.shape)

Repeated input: 1021
Repeated input + instruction: 148
Same examples: 148
(30536, 5)


In [281]:
tasks_values = train_df["task"].value_counts()
print(tasks_values)

task
simplification     9890
paraphrase         9599
ner                9500
masked_sentence    1547
Name: count, dtype: int64


In [279]:
def wiki_prompt_formation(record):
    input, instruction, task = record["input"], record["instruction"], record["task"]
    input = re.sub(r" {2,}", " ", input)
    instruction = re.sub(r" {2,}", " ", instruction)

    prompt = (
        f"Завдання: {task}\n"
        f"{input}\n"
        f"Інструкції: \n{instruction}\n"
    )

    return prompt, record["output"]


In [280]:
prompt, output = wiki_prompt_formation(row)

print("Prompt:\n", prompt)
print("Output:\n", output)

Prompt:
 Завдання: ner
Геогра́фія, або заст. земле́пис (від  — опис Землі; де  — Земля і  — писати, описувати) — система наук, що вивчає географічну оболонку Землі (епігеосферу), її просторову природну і соціально-економічну різноманітність, господарство і населення планети, окремих її регіонів та країн, а також зв'язки між природним середовищем і діяльністю людини. В сучасному розумінні поняття «географія» заміщено поняттям «географічні науки».
Інструкції: 
Витягни всі згадки про людей, місця, компанії та дати з тексту

Output:
 Люди: людина
Місця: Земля, регіони, країни
Компанії: —
Дати: —
